# Finding places

The feed index lists the world's GTFS (General Transit Feed Specification)
feeds and the places they serve: cities, metros, regions and countries. This
page shows how to install the index, look up a place and move between the
places that contain it.

## The feed index

For each place, the feed index records which feeds serve it and at which
**tier**: `local`, `regional`, `national` or `international` service. Install
the newest snapshot of the index once:

```python
import transitio

transitio.index.refresh()
```

The index is published as versioned snapshots by
[transitio-index](https://github.com/transitio-dev/transitio-index). With
`transitio`, you can install, list and pin these snapshots:

- `transitio.index.refresh()` installs the newest snapshot your version of
  `transitio` reads.
- `transitio.index.installed()` lists the snapshots installed on your
  computer.
- `transitio.index.use(snapshot_id)` pins one snapshot for the running Python
  process, for example to repeat an analysis with the same index. The
  `TRANSITIO_INDEX_SNAPSHOT` environment variable does the same.

Queries read the pinned snapshot, or the newest installed one when none is
pinned.

The first refresh downloads about 420 MB, which takes about 35 seconds at
100 Mbit/s or 3 minutes at 20 Mbit/s. Unpacking and checking the snapshot
takes about 10 seconds more. The refresh prints the download's progress to
stderr; `refresh(progress=False)` turns this off. A snapshot takes about
550 MB on disk. The cache keeps the three newest snapshots plus a pinned one,
up to about 2.2 GB in total, and a refresh needs about 1 GB of free space
while it runs.

## Look up a place

Let's start by looking up Augsburg:

In [1]:
import transitio

augsburg = transitio.place("Augsburg")
augsburg

Place(tp_56, city, 'Augsburg')

A qualifier after a comma names the region or country that holds the place:

In [2]:
transitio.place("London, Ontario")

Place(tp_42820, city, 'London')

`places` lists every place a name matches, starting with the one `place`
returns. `suggest` completes a name as you type it, searching names,
translations and aliases:

In [3]:
transitio.places("Augsburg")[:5]

[Place(tp_56, city, 'Augsburg'),
 Place(tp_132342, metro, 'Augsburg'),
 Place(tp_137435, metro, 'Augsburg'),
 Place(tp_132676, metro, 'Augsburg'),
 Place(tp_4, region, 'Augsburg (district)')]

In [4]:
transitio.suggest("augs")[:3]

[Suggestion(place=Place(tp_56, city, 'Augsburg'), label='Augsburg', matched='Augsburg', source='name'),
 Suggestion(place=Place(tp_132342, metro, 'Augsburg'), label='Augsburg', matched='Augsburg', source='name'),
 Suggestion(place=Place(tp_137435, metro, 'Augsburg'), label='Augsburg', matched='Augsburg', source='name')]

### How a name is resolved

A name can match several places. `place` then picks one by the rules below,
and raises `AmbiguousPlaceError` where none of them decides, as for
Springfield or Victoria. The reference of {func}`transitio.index.place`
describes the rules in full.

- **A large city wins first.** A city of at least 200,000 people that carries
  the name as its name or in English wins when it has more than twice the
  population of every other place of that name with a recorded population,
  whatever their feeds or labels: "Lima" is Lima, Peru, not Lima, Ohio. This
  rule does not apply where a region or country of the name contains no city
  of that name ("Victoria" is both a city in British Columbia and an
  Australian state), or where a city of the name without a recorded
  population is known at least as widely (San Jose, California).
- **A city wins over the places named after it.** A city wins over the metros
  in its country that carry its name, and over a same-named area containing
  it that runs much the same service: "Augsburg" is the city, not its three
  metros, and "Helsinki" is the city, not the Helsinki sub-region. It also
  wins over the areas in its country that list its name only as an alias:
  "Taipei" is the city, not New Taipei or Taiwan.
- **Without a city, a region or country of the name takes its place:**
  "Istanbul" is the province, not its metro.
- **A place's own names come first.** A place's own names are its name and
  its labels in its country's languages and in English: "München" is Munich.
  A place that carries a name only as a label in another language does not
  compete with them, so a place abroad labelled "Buenos Aires" in another
  language is no rival to the Argentine capital. Likewise, where a place
  carries a name as its name or in English, a place that carries it only in
  another of its own languages does not compete: "Bergen", Dutch for Mons in
  Belgium, is no rival to the towns named Bergen. A place known far more
  widely still competes, so "Meksyk", Polish for Mexico, stays ambiguous
  rather than naming a place in Poland.
- **Feeds, then how widely a place is known, decide the rest.** Other places
  sharing a name are decided by the only exact match or by a clear lead in
  feeds. Feed counts reflect how well each country's feeds are catalogued, so
  a lead in feeds does not decide against a place of that name abroad whose
  name is recorded in far more languages. Where feeds do not decide, the
  place of the name recorded in far more languages than every other, at home
  or abroad, wins: "Moscow" is Moscow, Russia, however many more feeds Moscow,
  Idaho, has, and "Cali, Colombia" is the city rather than a lesser-known
  place there.

Only a full name resolves. A partial name such as `"Augs"` raises
`PlaceNotFoundError`: its `candidates` hold the places the name partly
matches, and `suggest` completes it.

### Narrow down a lookup

You can narrow a lookup in three ways:

- **A qualifier after a comma** names the region or country that holds the
  place: `"London, Ontario"`, `"London, Canada"` or `"City of London, UK"`.
  The qualifier is matched against the names, translations and aliases of the
  place's region and country, so codes such as `UK` or `USA` work. A town or
  other containing place counts only when no region or country holds a place
  of the name, so `"Copenhagen, Denmark"` is in Denmark, not in the town of
  Denmark, New York.
- **`kind`** (`"city"`, `"metro"`, `"region"` or `"country"`) restricts the
  lookup to one kind of place.
- **A Wikidata id or the index's own id** picks one place (see Place ids
  below).

A name that itself contains a comma, such as the alias
`"Queen's Park, Greater London"`, still matches as written.

Each metro definition (see Move around the hierarchy below) names a metro
after its core city. When one metro exists under several definitions, so
that the metros share member places, `place` answers with the first of
`functional urban area`, `metropolitan statistical area`,
`metropolitan region` and `city-region (FAO)`. So
`transitio.place("Stockholm", kind="metro")` is Stockholm's functional urban
area, and `definition="metropolitan region"` picks another definition. A US
metropolitan statistical area is named after its principal cities, so a
city's name matches it only in part; the city's `metros` lists it.

## Place ids

Every place in the index has its own id, a `tp_<n>` that never changes and
is never reused. The index also keeps the external ids each place carries,
such as its Wikidata id. Let's look up Helsinki by its Wikidata id:

In [5]:
helsinki = transitio.place("Q1757")   # by Wikidata id, own id or name
print(helsinki.id)
print(helsinki.wikidata_id)    # None for a place without a Wikidata item
print(helsinki.concordances)
print(helsinki.former_ids)     # ids merged into this place; each still resolves to it

tp_9307
Q1757
{'overture': ['624768bb-b776-415a-90b0-ac19e9a35a03'], 'wikidata': ['Q1757']}
[]


Notice that `transitio.place("Q1757")` finds Helsinki through its
**concordances**, the ids the place carries in other datasets, here Overture
Maps and Wikidata. A Wikidata id (QID) of a place that was merged into
another place resolves to the place it was merged into, and
`transitio.place("tp_…")` resolves a former `tp_` id to the place that
replaced it.

## Move around the hierarchy

Every place knows the places that contain it. `subtype` is the place's level
in its source data, and `ancestors` lists the region and country that
contain it:

In [6]:
print(augsburg.subtype)
print(augsburg.ancestors)
print(augsburg.parent)     # .children goes the other way

locality
[Place(tp_192, region, 'Bavaria'), Place(tp_42, country, 'Germany')]
Place(tp_192, region, 'Bavaria')


`metros` lists the metros the place belongs to, and `delineations()` lists
every area the place is part of, with its relation to the place:

In [7]:
augsburg.metros

[Place(tp_132342, metro, 'Augsburg'),
 Place(tp_132508, metro, 'Munich'),
 Place(tp_137435, metro, 'Augsburg')]

In [8]:
augsburg.delineations()

[Delineation(relation='itself', kind='city', subtype='locality', place=Place(tp_56, city, 'Augsburg')),
 Delineation(relation='within', kind='region', subtype='region', place=Place(tp_192, region, 'Bavaria')),
 Delineation(relation='within', kind='country', subtype='country', place=Place(tp_42, country, 'Germany')),
 Delineation(relation='member of', kind='metro', subtype='city-region (FAO)', place=Place(tp_132508, metro, 'Munich')),
 Delineation(relation='member of', kind='metro', subtype='functional urban area', place=Place(tp_137435, metro, 'Augsburg')),
 Delineation(relation='member of', kind='metro', subtype='metropolitan region', place=Place(tp_132342, metro, 'Augsburg'))]

A city can belong to a metro of each of four definitions, told apart by
`subtype`:

- `functional urban area`: a city and its commuting zone, from the Eurostat
  Urban Audit;
- `metropolitan statistical area`: the metro areas of the US Census Bureau;
- `metropolitan region`: Eurostat's approximation of metro areas from NUTS-3
  regions, the smallest regions of the EU's statistical classification;
- `city-region (FAO)`: city regions worldwide, from the Food and Agriculture
  Organization of the United Nations.

`delineations()` returns the place itself, the region and country that
contain it, and these metros. Each item is a `Delineation` with four fields:
`relation`, `kind`, `subtype` and `place`.

With a place in hand, [Choosing feeds](choosing_feeds.ipynb) shows how to
list its feeds and pick the ones to fetch.